# Cibuco_Boriken — BirdCLEF+ 2026 Inference v8 (B2 + Perch Native)
**Team:** Cibuco_Boriken | **Ensemble:** EfficientNet-B2 + Perch v2 Native | **234 species**

Two maximally diverse models:
- **Google Perch v2** — Native bird classifier (14,795 species, trained on Xeno-Canto)
- **EfficientNet-B2** — CNN on mel spectrograms (206 training species)

In [ ]:
# ── Cell 1: Install dependencies ──
# Perch 2.0 requires TensorFlow 2.20.x according to the Kaggle model card.
import os, subprocess, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'tensorflow==2.20.0',
    'librosa',
    'audioread',
])
print('Dependencies installed ✅')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
print(f'TensorFlow version: {tf.__version__}')
assert tf.__version__.startswith('2.20.'), (
    'Perch v2 requires TensorFlow 2.20.x. If this still shows an older version, '
    'the Kaggle runtime did not switch packages cleanly and this SavedModel will not run there.'
)

In [ ]:
# ── Cell 2: Imports + config ──
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import librosa
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import time

# Paths
DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Model paths
PERCH_MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')
B2_MODEL_PATH   = Path('/kaggle/input/datasets/drakus74/efficientnet-b2-smartcrop/efficientnet_b2_SMARTCROP_BCE_20260315.pt')

# Config
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512
FMIN        = 50
FMAX        = 14000
TEMPERATURE = 0.3
BATCH_SIZE  = 32
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

ZERO_SHOT_PRIOR = 1.0 / 234

# Ensemble weights — Perch native is strong, give it more
W_PERCH = 0.7
W_B2    = 0.3

print(f'Device:      {DEVICE}')
print(f'Ensemble:    Perch={W_PERCH}, B2={W_B2}')
print(f'Temperature: {TEMPERATURE} (B2 only, Perch uses raw sigmoid)')
print('Config ready ✅')

In [ ]:
# ── Cell 3: Load species lists + Perch label mapping ──
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES     = taxonomy_df['primary_label'].tolist()
NUM_SPECIES = len(SPECIES)

train_df          = pd.read_csv(DATA_DIR / 'train.csv')
TRAIN_SPECIES     = sorted(train_df['primary_label'].unique().tolist())
NUM_TRAIN_SPECIES = len(TRAIN_SPECIES)
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}

# Perch label mapping — try labels.csv, fall back to any CSV in assets
label_file = PERCH_MODEL_DIR / 'assets' / 'labels.csv'
if not label_file.exists():
    assets_dir = PERCH_MODEL_DIR / 'assets'
    csvs = sorted(assets_dir.glob('*.csv')) if assets_dir.exists() else []
    assert len(csvs) > 0, f'No label CSV found in {assets_dir}'
    label_file = csvs[0]
label_df = pd.read_csv(label_file)
ebird_col = label_df.columns[0]
for col in ['ebird2021', 'ebird_code', 'species_code']:
    if col in label_df.columns:
        ebird_col = col
        break
perch_labels = label_df[ebird_col].astype(str).tolist()
perch_to_idx = {sp: i for i, sp in enumerate(perch_labels)}

mapped_perch   = [sp for sp in SPECIES if sp in perch_to_idx]
unmapped_perch = [sp for sp in SPECIES if sp not in perch_to_idx]

print(f'Taxonomy:    {NUM_SPECIES}')
print(f'Train:       {NUM_TRAIN_SPECIES}')
print(f'Perch label col: {ebird_col}')
print(f'Perch total: {len(perch_labels)}')
print(f'Perch mapped:{len(mapped_perch)}/{NUM_SPECIES}')
print(f'Unmapped:    {unmapped_perch}')

In [ ]:
# ── Cell 4: Load Perch v2 (TF) + smoke test ──
perch = tf.saved_model.load(str(PERCH_MODEL_DIR))
infer_fn = perch.signatures['serving_default']

test_wav = tf.constant(np.zeros((1, int(SAMPLE_RATE * WINDOW_SEC)), dtype=np.float32))
test_out = infer_fn(inputs=test_wav)
assert 'label' in test_out, f"Expected 'label' key, got: {list(test_out.keys())}"
print(f'Perch v2 loaded ✅  |  classes: {test_out[\"label\"].shape[-1]}  |  keys: {list(test_out.keys())}')

In [ ]:
# ── Cell 5: Load EfficientNet-B2 (PyTorch) ──
from torchvision.models import efficientnet_b2

model_b2 = efficientnet_b2(weights=None)
in_f = model_b2.classifier[1].in_features  # 1408
model_b2.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_f, NUM_TRAIN_SPECIES),
)
state_dict = torch.load(B2_MODEL_PATH, map_location=DEVICE, weights_only=True)
cleaned = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
model_b2.load_state_dict(cleaned)
model_b2 = model_b2.to(DEVICE)
model_b2.eval()
print(f'EfficientNet-B2 loaded ✅ ({in_f} → {NUM_TRAIN_SPECIES})')

In [ ]:
# ── Cell 6: Audio utilities ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_db = np.zeros_like(mel_db)
    return mel_db.astype(np.float32)

def mel_to_tensor(mel):
    t = torch.from_numpy(mel).unsqueeze(0)
    t = t.repeat(3, 1, 1)
    t = nnF.interpolate(
        t.unsqueeze(0), size=(224, 224),
        mode='bilinear', align_corners=False,
    ).squeeze(0)
    return t

print('Audio utilities ready ✅')

In [ ]:
# ── Cell 7: Parse sample submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'Sample submission: {len(sample_sub)} rows, {len(sample_sub.columns)} cols')

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

print(f'Soundscape files: {len(file_windows)}')
print('Row IDs parsed ✅')

In [ ]:
# ── Cell 8: B2 + Perch Native ensemble inference ──
soundscape_dir = DATA_DIR / 'test_soundscapes'
window_samples = int(WINDOW_SEC * SAMPLE_RATE)

all_rows = {}
t_start = time.time()
total_windows = sum(len(v) for v in file_windows.values())
processed = 0

for file_stem in tqdm(sorted(file_windows.keys()), desc='B2+Perch Native'):
    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            audio_path = candidate
            break

    if audio_path is None:
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    try:
        audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    # Collect all chunks for this file
    chunks = []
    row_ids_order = []
    for row_id, end_time in file_windows[file_stem]:
        start_sample = max(0, int((end_time - WINDOW_SEC) * SAMPLE_RATE))
        end_sample = int(end_time * SAMPLE_RATE)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        chunks.append(chunk)
        row_ids_order.append(row_id)

    # ── B2 batch inference (fast on CPU) ──
    mel_tensors = []
    for chunk in chunks:
        mel = audio_to_mel(chunk)
        mel_tensors.append(mel_to_tensor(mel))
    mel_batch = torch.stack(mel_tensors).to(DEVICE)

    all_probs_b2 = []
    for i in range(0, len(mel_batch), BATCH_SIZE):
        with torch.no_grad():
            logits = model_b2(mel_batch[i:i+BATCH_SIZE])
            probs = torch.sigmoid(logits / TEMPERATURE).cpu().numpy()
        all_probs_b2.append(probs)
    probs_b2 = np.vstack(all_probs_b2)  # (num_windows, NUM_TRAIN_SPECIES)

    # ── Perch native inference (per-window, TF) ──
    probs_perch_list = []
    for chunk in chunks:
        waveform = tf.constant(chunk[np.newaxis, :], dtype=tf.float32)
        output = infer_fn(inputs=waveform)
        logits_perch = output['label'].numpy().squeeze()  # (14795,)
        probs_p = 1.0 / (1.0 + np.exp(-logits_perch))    # sigmoid
        probs_perch_list.append(probs_p)
    probs_perch = np.stack(probs_perch_list)  # (num_windows, 14795)

    # ── Combine per-species ──
    for idx, row_id in enumerate(row_ids_order):
        row = {'row_id': row_id}
        for sp in SPECIES:
            # Perch native probability
            if sp in perch_to_idx:
                p_perch = float(probs_perch[idx, perch_to_idx[sp]])
            else:
                p_perch = ZERO_SHOT_PRIOR

            # B2 probability
            if sp in train_species_idx:
                p_b2 = float(probs_b2[idx, train_species_idx[sp]])
            else:
                p_b2 = ZERO_SHOT_PRIOR

            row[sp] = W_PERCH * p_perch + W_B2 * p_b2
        all_rows[row_id] = row

        processed += 1
        if processed % 100 == 0:
            elapsed = time.time() - t_start
            rate = processed / elapsed
            remaining = (total_windows - processed) / rate / 60
            print(f'  {processed}/{total_windows} | {elapsed/60:.1f}min elapsed | ~{remaining:.1f}min remaining')

elapsed = time.time() - t_start
print(f'\nTotal rows: {len(all_rows)} (expected: {len(sample_sub)})')
print(f'Total time: {elapsed/60:.1f} min ✅')

In [ ]:
# ── Cell 9: Generate submission.csv ──
rows_ordered = [all_rows[rid] for rid in sample_sub['row_id']]
submission_df = pd.DataFrame(rows_ordered)
submission_df = submission_df[sample_sub.columns]
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape} (expected: {sample_sub.shape})')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 10: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ B2+PERCH NATIVE ENSEMBLE VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')